In [5]:
%pip install "numpy==1.26.4" "scipy==1.15.3" "scikit-learn==1.6.1" joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 160.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 138.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()
print(dataset.data.shape)

(569, 30)


In [3]:
from sklearn.model_selection import train_test_split

X = dataset.data       # The 30 measurements for each record
y = dataset.target     # Diagnosis: 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

Training features: (455, 30)
Test features: (114, 30)


In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

logistic_model.fit(X_train, y_train)
y_pred = logistic_model.predict(X_test)

print("First 10 predictions:", y_pred[:10])
print("First 10 actual labels:", y_test[:10])

First 10 predictions: [0 1 0 1 0 1 1 0 0 0]
First 10 actual labels: [0 1 0 1 0 1 1 0 0 0]


In [7]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

print("First 10 tree predictions:", tree_pred[:10])
print("First 10 actual labels:   ", y_test[:10])

First 10 tree predictions: [0 1 0 1 0 1 1 0 0 0]
First 10 actual labels:    [0 1 0 1 0 1 1 0 0 0]


In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)

print("First 10 KNN predictions:", knn_pred[:10])
print("First 10 actual labels:  ", y_test[:10])

First 10 KNN predictions: [0 1 0 0 0 1 1 0 0 0]
First 10 actual labels:   [0 1 0 1 0 1 1 0 0 0]


In [11]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

print("First 10 Naive Bayes predictions:", nb_pred[:10])
print("First 10 actual labels:          ", y_test[:10])

First 10 Naive Bayes predictions: [0 1 0 1 0 1 1 0 0 0]
First 10 actual labels:           [0 1 0 1 0 1 1 0 0 0]


In [13]:
from sklearn.svm import SVC

svm_model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", probability=True, random_state=42)
)

svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)

print("First 10 SVM predictions:", svm_pred[:10])
print("First 10 actual labels:  ", y_test[:10])

First 10 SVM predictions: [0 1 0 1 0 1 1 0 0 0]
First 10 actual labels:   [0 1 0 1 0 1 1 0 0 0]


In [15]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "KNN": knn_model,
    "Gaussian Naive Bayes": nb_model,
    "SVM": svm_model
}

results = []

for name, model in models.items():
    predictions = model.predict(X_test)

    # Use the probability of malignant (label 0) for AUC.
    malignant_column = list(model.classes_).index(0)
    malignant_scores = model.predict_proba(X_test)[:, malignant_column]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(y_test == 0, malignant_scores),
        "Precision": precision_score(y_test, predictions, pos_label=0),
        "Recall": recall_score(y_test, predictions, pos_label=0),
        "F1": f1_score(y_test, predictions, pos_label=0),
        "MCC": matthews_corrcoef(y_test, predictions)
    })

comparison = pd.DataFrame(results)
display(comparison.round(4))

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623
1,Decision Tree,0.9123,0.9157,0.8478,0.9286,0.8864,0.8174
2,KNN,0.9561,0.9788,0.9512,0.9286,0.9398,0.9054
3,Gaussian Naive Bayes,0.9386,0.9878,0.9268,0.9048,0.9157,0.8676
4,SVM,0.9825,0.9950,0.9762,0.9762,0.9762,0.9623


In [17]:
from pathlib import Path
import joblib

model_folder = Path("model")
model_folder.mkdir(exist_ok=True)

for name, model in models.items():
    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, model_folder / filename)
    print("Saved:", model_folder / filename)

test_data = pd.DataFrame(X_test, columns=dataset.feature_names)
test_data["diagnosis"] = y_test  # 0 = malignant, 1 = benign
test_data.to_csv("test_data.csv", index=False)

print("Test CSV shape:", test_data.shape)
print("Files saved under:", Path.cwd())

Saved: model/logistic_regression.joblib
Saved: model/decision_tree.joblib
Saved: model/knn.joblib
Saved: model/gaussian_naive_bayes.joblib
Saved: model/svm.joblib
Test CSV shape: (114, 31)
Files saved under: /home/s2026ak05209/workspace


In [19]:
import numpy as np
from importlib.metadata import version

saved_csv = pd.read_csv("test_data.csv")
csv_features = saved_csv[list(dataset.feature_names)].to_numpy()

for name, original_model in models.items():
    filename = name.lower().replace(" ", "_") + ".joblib"
    saved_model = joblib.load(Path("model") / filename)

    matches = np.array_equal(
        saved_model.predict(csv_features),
        original_model.predict(X_test)
    )
    print(name, "PASS" if matches else "CHECK")

print("\nVersions for requirements.txt:")
for package in ["numpy", "scipy", "scikit-learn", "pandas", "joblib"]:
    print(f"{package}=={version(package)}")

Logistic Regression PASS
Decision Tree PASS
KNN PASS
Gaussian Naive Bayes PASS
SVM PASS

Versions for requirements.txt:
numpy==1.26.4
scipy==1.15.3
scikit-learn==1.6.1
pandas==2.2.3
joblib==1.4.2
